In [1]:
# ============================================================
# GO semantic similarity reduction using rrvgo
# For GO Biological Process, Molecular Function, Cellular Component
# ============================================================

# Install packages if not already installed
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")

packages_bioc <- c("rrvgo", "org.Hs.eg.db", "GO.db", "GOSemSim")
for (pkg in packages_bioc) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    BiocManager::install(pkg, update = FALSE, ask = FALSE)
  }
}

packages_cran <- c("readr", "dplyr", "stringr", "ggplot2", "openxlsx")
for (pkg in packages_cran) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Installing package(s) 'BiocVersion', 'rrvgo'

also installing the dependencies ‘XVector’, ‘Seqinfo’, ‘R.oo’, ‘R.methodsS3’, ‘png’, ‘Biostrings’, ‘RcppTOML’, ‘here’, ‘RcppEigen’, ‘R.utils’, ‘yulab.utils’, ‘BiocGenerics’, ‘Biobase’, ‘IRanges’, ‘RSQLite’, ‘S4Vectors’, ‘KEGGREST’, ‘colorspace’, ‘gridBase’, ‘igraph’, ‘NLP’, ‘slam’, ‘BH’, ‘reticulate’, ‘RSpectra’, ‘GOSemSim’, ‘AnnotationDbi’, ‘GO.db’, ‘pheatmap’, ‘ggrepel’, ‘treemap’, ‘tm’, ‘wordcloud’, ‘umap’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com


In [2]:
library(rrvgo)
library(org.Hs.eg.db)
library(GO.db)
library(GOSemSim)
library(readr)
library(dplyr)
library(stringr)
library(ggplot2)
library(openxlsx)

# ------------------------------------------------------------
# 1. Set file paths
# ------------------------------------------------------------



Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view w

In [9]:
bp_file <- "/content/GO_Biological_Process_2026_table.txt"
mf_file <- "/content/GO_Molecular_Function_2026_table.txt"
cc_file <- "/content/GO_Cellular_Component_2026_table.txt"

output_dir <- "GO_simplified_results"
dir.create(output_dir, showWarnings = FALSE)

# ------------------------------------------------------------
# 2. Function to simplify GO table
# ------------------------------------------------------------

simplify_go_table <- function(file_path, ontology = "BP", output_prefix = "GO_BP") {

  # Read Enrichr/richR table
  go_df <- read_tsv(file_path, show_col_types = FALSE)

  # Extract GO IDs from Term column
  go_df <- go_df %>%
    mutate(
      GO_ID = str_extract(Term, "GO:\\d+"),
      Term_clean = str_remove(Term, "\\s*\\(GO:\\d+\\)")
    ) %>% # This was a bug: \) and (
    filter(!is.na(GO_ID))

  # Use -log10 adjusted p-value as score
  # Smaller adjusted p-value = stronger term
  go_df <- go_df %>%
    mutate(
      score = -log10(`Adjusted P-value`)
    )

  # In case adjusted p-value is 0 or missing
  go_df$score[is.infinite(go_df$score)] <- max(go_df$score[is.finite(go_df$score)], na.rm = TRUE) + 1
  go_df$score[is.na(go_df$score)] <- -log10(go_df$`P-value`[is.na(go_df$score)])

  # Named score vector required by rrvgo
  scores <- go_df$score
  names(scores) <- go_df$GO_ID

  # Remove duplicate GO IDs, keep best score
  scores <- tapply(scores, names(scores), max)

  # Calculate semantic similarity
  simMatrix <- calculateSimMatrix(
    names(scores),
    orgdb = "org.Hs.eg.db",
    ont = ontology,
    method = "Rel"
  )

  # Reduce redundant terms
  reducedTerms <- reduceSimMatrix(
    simMatrix = simMatrix,
    scores = scores,
    threshold = 0.7,
    orgdb = "org.Hs.eg.db"
  )

  # Join original details
  reducedTerms_out <- reducedTerms %>%
    as.data.frame() %>%
    tibble::rownames_to_column("GO_ID") %>%
    left_join(go_df, by = "GO_ID")

  # Save full reduced table
  write.xlsx(
    reducedTerms_out,
    file = file.path(output_dir, paste0(output_prefix, "_rrvgo_reduced_terms.xlsx")),
    overwrite = TRUE
  )

  # Only create scatter plots if there are at least 3 terms in reducedTerms
  if (nrow(reducedTerms) >= 3) {
    # Save scatter plot
    pdf(file.path(output_dir, paste0(output_prefix, "_semantic_similarity_scatter.pdf")),
        width = 9, height = 7)
    scatterPlot(
      simMatrix,
      reducedTerms,
      labelSize = 3
    )
    dev.off()

    png(file.path(output_dir, paste0(output_prefix, "_semantic_similarity_scatter.png")),
        width = 3000, height = 2400, res = 300)
    scatterPlot(
      simMatrix,
      reducedTerms,
      labelSize = 3
    )
    dev.off()
  } else {
    message(paste0("Skipping scatter plots for ", output_prefix, " due to insufficient terms (less than 3)."))
  }

  # Save treemap (treemapPlot can handle 1 or 2 terms, though may not be very informative)
  pdf(file.path(output_dir, paste0(output_prefix, "_treemap.pdf")),
      width = 10, height = 8)
  treemapPlot(reducedTerms)
  dev.off()

  png(file.path(output_dir, paste0(output_prefix, "_treemap.png")),
      width = 3200, height = 2600, res = 300)
  treemapPlot(reducedTerms)
    dev.off()

  return(reducedTerms_out)
}

# ------------------------------------------------------------
# 3. Run simplification for BP, MF, CC
# ------------------------------------------------------------

bp_reduced <- simplify_go_table(bp_file, ontology = "BP", output_prefix = "GO_BP")
mf_reduced <- simplify_go_table(mf_file, ontology = "MF", output_prefix = "GO_MF")
cc_reduced <- simplify_go_table(cc_file, ontology = "CC", output_prefix = "GO_CC")

# ------------------------------------------------------------
# 4. Export all simplified results into one Excel workbook
# ------------------------------------------------------------

write.xlsx(
  list(
    GO_BP_reduced = bp_reduced,
    GO_MF_reduced = mf_reduced,
    GO_CC_reduced = cc_reduced
  ),
  file = file.path(output_dir, "All_GO_rrvgo_simplified_results.xlsx"),
  overwrite = TRUE
)

cat("GO simplification completed. Results saved in:", output_dir)


preparing gene to GO mapping data...

preparing IC data...

'select()' returned 1:many mapping between keys and columns

preparing gene to GO mapping data...

preparing IC data...

'select()' returned 1:many mapping between keys and columns

preparing gene to GO mapping data...

preparing IC data...

'select()' returned 1:many mapping between keys and columns

Skipping scatter plots for GO_CC due to insufficient terms (less than 3).



GO simplification completed. Results saved in: GO_simplified_results

In [10]:
# Psoriasis-relevant biological keyword list
psoriasis_keywords <- c(
  "immune", "inflammatory", "inflammation", "cytokine",
  "interferon", "tnf", "il-17", "il17", "il-23", "il23",
  "jak", "stat", "nf-kappa", "nfkb",
  "keratinocyte", "epiderm", "skin", "epithelial",
  "wound", "barrier", "cornification",
  "extracellular matrix", "matrix", "collagen",
  "angiogenesis", "vascular",
  "oxidative stress", "reactive oxygen",
  "cell cycle", "proliferation", "differentiation",
  "apoptosis", "death receptor",
  "lipid", "ppar", "fatty acid",
  "wnt", "mapk", "ras",
  "mrna", "rna processing", "splicing",
  "glycosylation", "dolichol"
)

pattern <- paste(psoriasis_keywords, collapse = "|")

filter_relevant_terms <- function(df) {
  df %>%
    mutate(
      Term_lower = str_to_lower(Term),
      Psoriasis_relevance = ifelse(str_detect(Term_lower, pattern), "Relevant", "Review/Exclude")
    ) %>%
    arrange(Psoriasis_relevance, `Adjusted P-value`)
}

# Load original dataframes
go_bp <- read_tsv(bp_file, show_col_types = FALSE)
go_mf <- read_tsv(mf_file, show_col_types = FALSE)
go_cc <- read_tsv(cc_file, show_col_types = FALSE)

go_bp_filtered <- filter_relevant_terms(go_bp)
go_mf_filtered <- filter_relevant_terms(go_mf)
go_cc_filtered <- filter_relevant_terms(go_cc)


write.xlsx(
  list(
    GO_BP_filtered = go_bp_filtered,
    GO_MF_filtered = go_mf_filtered,
    GO_CC_filtered = go_cc_filtered
  ),
  file = "Psoriasis_relevance_filtered_enrichment_results.xlsx",
  overwrite = TRUE
)